# IPSA CSBS-CG Original vs Corrected Scoring Audit

This notebook performs a **read-only** audit of the IPSA REDCap CSBS Caregiver Questionnaire (CSBS-CG) scoring fields. It does **not** modify any REDCap data.

**What this notebook does:**

1. **Compares** the original and corrected REDCap calculated fields for Speech, Symbolic, and Total weighted raw scores.
2. **Independently recomputes** all CSBS-CG scores from source items using validated scoring logic.
3. **Identifies why** discrepancies occur (Other-field inclusion, rounding hierarchy, or both).
4. **Determines which fields** align with the validated scoring logic.
5. **Shows downstream effects** on standard scores and percentile ranks where validated norms are available (23–24 month age band only).

> ⚠️ This is a **read-only audit**. No REDCap records are created, modified, or deleted.

## 1. Imports and Display Settings

In [ ]:
import os
import re
import math
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

# Readable pandas output
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

## 2. Secure Configuration

The API token is read from the environment variable `IPSA_API_TOKEN`. It is never displayed, logged, or exported.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
API_URL = "https://redcap.research.sc.edu/api/"
API_TIMEOUT = 60
TARGET_FORM = "csbs_caregiver"                # confirmed from metadata below
EVENT_FILTER = None                           # None = audit all CSBS-CG events
EXPORT_RESULTS = False                        # set True to save CSVs
EXPECTED_PROJECT_TITLE_FRAGMENT = "IPSA"      # safety check

# ── Token ─────────────────────────────────────────────────────────────────────
IPSA_API_TOKEN = os.getenv("IPSA_API_TOKEN")
if not IPSA_API_TOKEN:
    raise EnvironmentError(
        "IPSA_API_TOKEN is not set. Please set it securely in your environment "
        "before running this notebook (e.g. via .env or a secrets manager)."
    )
print("✅ IPSA_API_TOKEN loaded from environment.")

## 3. Minimal REDCap API Client

A single reusable POST helper for all read-only API calls. The token is sent in POST data only—never in URLs, logs, or error messages.

In [ ]:
def redcap_post(content, **kwargs):
    """POST to REDCap API. Token is never exposed in errors or logs."""
    data = {"token": IPSA_API_TOKEN, "content": content,
            "format": "json", "returnFormat": "json"}
    # Handle list-valued parameters (e.g. forms, fields)
    for key, value in kwargs.items():
        if isinstance(value, (list, tuple)):
            for v in value:
                data.setdefault(f"{key}[]", [])
                if isinstance(data[f"{key}[]"], list):
                    # requests handles list values as repeated keys
                    pass
            # Use the list form directly
            data[f"{key}[]"] = list(value)
        else:
            data[key] = value
    response = requests.post(API_URL, data=data, timeout=API_TIMEOUT)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(f"REDCap API error: {payload['error']}")
    return payload


# ── Fetch project info ────────────────────────────────────────────────────────
project_info = redcap_post("project")
project_title = project_info.get("project_title", "")
is_longitudinal = bool(project_info.get("is_longitudinal"))
print(f"Project: {project_title}")
print(f"Longitudinal: {is_longitudinal}")

assert EXPECTED_PROJECT_TITLE_FRAGMENT in project_title, (
    f"Project title '{project_title}' does not contain "
    f"'{EXPECTED_PROJECT_TITLE_FRAGMENT}'. Wrong project?"
)
print(f"✅ Project title confirmed: contains '{EXPECTED_PROJECT_TITLE_FRAGMENT}'")

# ── Fetch metadata ───────────────────────────────────────────────────────────
all_metadata = redcap_post("metadata")
record_id_field = all_metadata[0]["field_name"]
print(f"Record ID field: {record_id_field}")

# ── Fetch events (longitudinal) ──────────────────────────────────────────────
events_meta = redcap_post("event") if is_longitudinal else []
event_labels = {e["unique_event_name"]: e["event_name"] for e in events_meta}
print(f"Events: {len(event_labels)}")

# ── Filter CSBS-CG metadata ─────────────────────────────────────────────────
csbs_meta = [m for m in all_metadata if m.get("form_name") == TARGET_FORM]
confirmed_form = csbs_meta[0]["form_name"] if csbs_meta else None
assert confirmed_form == TARGET_FORM, (
    f"Expected form '{TARGET_FORM}' not found in metadata. "
    f"Available forms: {sorted(set(m['form_name'] for m in all_metadata))}"
)
print(f"✅ CSBS-CG form confirmed: {confirmed_form} ({len(csbs_meta)} fields)")

## 4. Dynamic Metadata Inventory

We resolve **original** and **corrected** scoring fields from live REDCap metadata—not from hardcoded names. Fields whose names or labels contain "Other" are flagged.

In [ ]:
# Build metadata lookup
meta_by_name = {m["field_name"]: m for m in csbs_meta}
calc_fields = {m["field_name"]: m for m in csbs_meta if m.get("field_type") == "calc"}

# ── Identify "Other" fields ──────────────────────────────────────────────────
other_fields = set()
for m in csbs_meta:
    name_lower = m["field_name"].lower()
    label_lower = m.get("field_label", "").lower()
    if "other" in name_lower or re.search(r"\bother\b", label_lower):
        other_fields.add(m["field_name"])

print(f'Fields matching "Other" pattern: {sorted(other_fields)}')

# ── Resolve original/corrected pairs dynamically ─────────────────────────────
# We search calc fields by name patterns and label matching.

PAIR_SPEC = {
    "Speech composite": {
        "original_candidates": ["csbs_speechcomposite"],
        "corrected_candidates": ["csbscg_speechrawcorr"],
        "label_hint": "speech composite",
    },
    "Words cluster": {
        "original_candidates": ["csbs_words"],
        "corrected_candidates": ["csbscg_wordsawcorr", "csbscg_wordsrawcorr"],
        "label_hint": "words",
    },
    "Understanding cluster": {
        "original_candidates": ["csbs_understanding"],
        "corrected_candidates": ["csbscg_understandrawcorr", "csbscg_understandingrawcorr"],
        "label_hint": "understanding",
    },
    "Symbolic composite": {
        "original_candidates": ["csbs_symboliccomposite"],
        "corrected_candidates": ["csbscg_ssymbolicrawcorr", "csbscg_symbolicrawcorr"],
        "label_hint": "symbolic composite",
    },
    "Total weighted raw score": {
        "original_candidates": ["cbs_totalscore", "cbscg_totalscore"],
        "corrected_candidates": ["csbscg_totalscorerawcorr"],
        "label_hint": "total",
    },
}

def resolve_field(candidates, calc_fields_dict, label_hint):
    """Resolve a field from candidate names; returns (field_name, meta_dict) or raises."""
    found = []
    for c in candidates:
        if c in calc_fields_dict:
            found.append((c, calc_fields_dict[c]))
    if len(found) == 1:
        return found[0]
    if len(found) > 1:
        raise ValueError(
            f"Ambiguous resolution for '{label_hint}': "
            f"multiple candidates found: {[f[0] for f in found]}"
        )
    # Fallback: search all calc fields by label
    for name, m in calc_fields_dict.items():
        if label_hint.lower() in m.get("field_label", "").lower():
            found.append((name, m))
    if len(found) == 1:
        return found[0]
    if len(found) > 1:
        raise ValueError(
            f"Ambiguous label-based resolution for '{label_hint}': "
            f"{[f[0] for f in found]}"
        )
    raise ValueError(
        f"Cannot resolve field for '{label_hint}'. "
        f"Candidates tried: {candidates}. "
        f"Available calc fields: {sorted(calc_fields_dict.keys())}"
    )

resolved_pairs = {}
for measure, spec in PAIR_SPEC.items():
    orig_name, orig_meta = resolve_field(
        spec["original_candidates"], calc_fields, spec["label_hint"]
    )
    corr_name, corr_meta = resolve_field(
        spec["corrected_candidates"], calc_fields, spec["label_hint"] + " corrected"
    )
    resolved_pairs[measure] = {
        "original_field": orig_name,
        "original_label": orig_meta.get("field_label", ""),
        "corrected_field": corr_name,
        "corrected_label": corr_meta.get("field_label", ""),
    }

# ── Also resolve cluster-only fields (no corrected pair) ────────────────────
CLUSTER_FIELDS = {
    "Emotion and eye gaze": "csbs_emotionandeyegaze",
    "Communication": "csbs_communication",
    "Gestures": "csbs_gestures",
    "Sounds": "csbs_sounds",
    "Social composite": "csbs_socialcomposite",
}

for measure, field_name in CLUSTER_FIELDS.items():
    if field_name not in calc_fields:
        print(f"⚠️  Cluster field '{field_name}' not found in metadata.")

# ── Display Field Resolution Table ──────────────────────────────────────────
resolution_rows = []
for measure, pair in resolved_pairs.items():
    resolution_rows.append({
        "Measure": measure,
        "Original field": pair["original_field"],
        "Original label": pair["original_label"],
        "Corrected field": pair["corrected_field"],
        "Corrected label": pair["corrected_label"],
        "Resolution status": "✅ Resolved from metadata",
    })

field_resolution_df = pd.DataFrame(resolution_rows)
display(Markdown("### Field Resolution Table"))
display(field_resolution_df)

## 5. Formula-Difference Audit

For each original/corrected pair, we extract the REDCap calculation, compare referenced fields, and classify differences. The key hypothesis is that original formulas include "Other" response fields (e.g., `csbs26_other`, `csbs32_other`) while corrected formulas exclude them.

In [ ]:
def extract_field_refs(formula):
    """Extract all [field] references from a REDCap formula."""
    return set(re.findall(r"\[(\w+)\]", formula or ""))


def normalize_formula(formula):
    """Strip whitespace for structural comparison."""
    return re.sub(r"\s+", "", formula or "")


def detect_rounding_structure(formula):
    """Extract the rounding/grouping structure of a formula."""
    # Count roundup() calls and their nesting
    clean = normalize_formula(formula)
    roundup_count = clean.count("roundup(")
    # Check if it sums rounded sub-expressions (composite hierarchy)
    has_composite_structure = bool(re.search(r"\(roundup\(.*\)\)\+\(roundup\(", clean))
    return {"roundup_calls": roundup_count, "composite_hierarchy": has_composite_structure}


formula_diff_rows = []
formula_details = {}

for measure, pair in resolved_pairs.items():
    orig_field = pair["original_field"]
    corr_field = pair["corrected_field"]
    orig_formula = calc_fields[orig_field].get("select_choices_or_calculations", "")
    corr_formula = calc_fields[corr_field].get("select_choices_or_calculations", "")

    orig_refs = extract_field_refs(orig_formula)
    corr_refs = extract_field_refs(corr_formula)

    only_in_original = orig_refs - corr_refs
    only_in_corrected = corr_refs - orig_refs

    # Flag "Other" fields
    other_in_original = only_in_original & other_fields
    other_in_corrected = only_in_corrected & other_fields

    # Rounding structure comparison
    orig_rounding = detect_rounding_structure(orig_formula)
    corr_rounding = detect_rounding_structure(corr_formula)
    rounding_differs = orig_rounding != corr_rounding

    # Interpretation
    if other_in_original and rounding_differs:
        interpretation = "Both: Other field(s) in original formula AND different rounding structure"
    elif other_in_original:
        interpretation = "Original includes Other-response field(s) not in corrected"
    elif rounding_differs:
        interpretation = "Rounding/composite hierarchy differs"
    elif only_in_original or only_in_corrected:
        interpretation = "Field set differs (non-Other)"
    else:
        interpretation = "Formulas are structurally equivalent"

    formula_diff_rows.append({
        "Measure": measure,
        "Only in original": ", ".join(sorted(only_in_original)) or "—",
        "Only in corrected": ", ".join(sorted(only_in_corrected)) or "—",
        '"Other" field involved': "Yes — " + ", ".join(sorted(other_in_original)) if other_in_original else "No",
        "Rounding structure differs": "Yes" if rounding_differs else "No",
        "Interpretation": interpretation,
    })

    formula_details[measure] = {
        "original_formula": orig_formula,
        "corrected_formula": corr_formula,
        "original_refs": sorted(orig_refs),
        "corrected_refs": sorted(corr_refs),
    }

formula_diff_df = pd.DataFrame(formula_diff_rows)
display(Markdown("### Formula Difference Table"))
display(formula_diff_df)

### Detailed Formula Comparison

The full formulas are shown below for reference. Scroll horizontally if needed.

In [ ]:
for measure, details in formula_details.items():
    display(Markdown(f"#### {measure}"))
    display(Markdown(f"**Original** (`{resolved_pairs[measure]['original_field']}`):\n```\n{details['original_formula']}\n```"))
    display(Markdown(f"**Corrected** (`{resolved_pairs[measure]['corrected_field']}`):\n```\n{details['corrected_formula']}\n```"))
    only_orig = set(details["original_refs"]) - set(details["corrected_refs"])
    if only_orig:
        display(Markdown(f"Fields only in original: **{', '.join(sorted(only_orig))}**"))
    print()

## 6. Validated Scoring Constants and Norms

These constants are taken directly from the validated reference implementation (`redcap_csbs_edi_scoring_audit.py`) and the supplied CSBS-CG norms PDF (23–24 month age band). They are **not** interpolated or extrapolated.

In [ ]:
# ── Scale-level standard score → percentile (scales/composites) ──────────────
CG_SCALE_PERCENTILE = {
    17: 99, 16: 98, 15: 95, 14: 91, 13: 84, 12: 75, 11: 63,
    10: 50, 9: 37, 8: 25, 7: 16, 6: 9, 5: 5, 4: 2, 3: 1,
}

# ── Total standard score → percentile ────────────────────────────────────────
CG_TOTAL_PERCENTILE = {
    135: 99, 134: 99, 133: 99, 132: 98, 131: 98, 130: 98,
    129: 97, 128: 97, 127: 96, 126: 96, 125: 95, 124: 95,
    123: 94, 122: 93, 121: 92, 120: 91, 119: 90, 118: 89,
    117: 87, 116: 86, 115: 84, 114: 83, 113: 81, 112: 79,
    111: 77, 110: 75, 109: 73, 108: 70, 107: 68, 106: 66,
    105: 63, 104: 61, 103: 58, 102: 55, 101: 53, 100: 50,
    99: 47, 98: 45, 97: 42, 96: 40, 95: 37, 94: 35, 93: 32,
    92: 30, 91: 27, 90: 25, 89: 23, 88: 21, 87: 19, 86: 18,
    85: 16, 84: 14, 83: 13, 82: 12, 81: 10, 80: 9, 79: 8,
    78: 7, 77: 6, 76: 6, 75: 5, 74: 4, 73: 4, 72: 3, 71: 3,
    70: 2, 69: 2, 68: 2, 67: 1, 66: 1, 65: 1,
}

# ── Raw score → standard score (23–24 months) ───────────────────────────────
# Each tuple: (raw_min, raw_max, standard_score)
CG_NORMS_23_24 = {
    "csbs_emotionandeyegaze": [(16,16,17),(15,15,14),(14,14,11),(13,13,9),(12,12,7),(11,11,5),(10,10,4),(0,9,3)],
    "csbs_communication": [(20,20,17),(19,19,12),(18,18,10),(16,17,9),(15,15,8),(12,14,6),(10,11,5),(9,9,4),(0,8,3)],
    "csbs_gestures": [(12,12,17),(11,11,9),(10,10,6),(9,9,5),(7,8,4),(0,6,3)],
    "csbs_sounds": [(16,16,17),(15,15,11),(14,14,10),(12,13,9),(11,11,8),(9,10,7),(7,8,6),(6,6,5),(0,5,3)],
    "csbs_words": [(24,24,17),(23,23,12),(20,22,11),(17,19,10),(13,16,9),(10,12,8),(8,9,7),(4,7,6),(3,3,5),(2,2,4),(0,1,3)],
    "csbs_understanding": [(24,24,17),(23,23,10),(22,22,9),(20,21,8),(15,19,7),(12,14,6),(7,11,5),(6,6,4),(0,5,3)],
    "csbs_objectuse": [(27,27,17),(25,26,12),(23,24,11),(22,22,10),(20,21,9),(18,19,8),(16,17,7),(14,15,6),(13,13,5),(10,12,4),(0,9,3)],
    "csbs_socialcomposite": [(48,48,17),(47,47,15),(46,46,14),(45,45,12),(43,44,11),(42,42,10),(40,41,9),(39,39,8),(38,38,7),(35,37,6),(34,34,5),(27,33,4),(0,26,3)],
    "csbs_speechcomposite": [(40,40,17),(39,39,13),(37,38,12),(33,36,11),(30,32,10),(25,29,9),(21,24,8),(16,20,7),(12,15,6),(10,11,5),(9,9,4),(0,8,3)],
    "csbs_symboliccomposite": [(51,51,17),(50,50,13),(49,49,12),(46,48,11),(44,45,10),(41,43,9),(36,40,8),(34,35,7),(27,33,6),(24,26,5),(17,23,4),(0,16,3)],
    "total": [(139,139,135),(138,138,131),(136,137,125),(135,135,124),(134,134,118),(133,133,115),(132,132,114),(130,131,112),(129,129,111),(128,128,110),(127,127,108),(126,126,106),(125,125,105),(124,124,104),(122,123,103),(121,121,102),(120,120,101),(119,119,100),(118,118,99),(117,117,98),(116,116,97),(115,115,96),(114,114,95),(112,113,94),(110,111,93),(109,109,92),(106,108,91),(104,105,90),(102,103,89),(100,101,88),(99,99,87),(98,98,86),(96,97,85),(94,95,84),(93,93,83),(88,92,82),(87,87,81),(84,86,80),(82,83,79),(80,81,78),(79,79,77),(77,78,76),(75,76,75),(74,74,74),(73,73,73),(72,72,72),(71,71,71),(61,70,70),(59,60,69),(56,58,68),(53,55,67),(50,52,66),(0,49,65)],
}

# ── Maximum raw scores ───────────────────────────────────────────────────────
CG_RAW_MAX = {
    "csbs_emotionandeyegaze": 16, "csbs_communication": 20, "csbs_gestures": 12,
    "csbs_sounds": 16, "csbs_words": 24, "csbs_understanding": 24,
    "csbs_objectuse": 27, "csbs_socialcomposite": 48, "csbs_speechcomposite": 40,
    "csbs_symboliccomposite": 51, "total": 139,
}

# ── Norm field mapping: raw_key → (display_label, ss_field, percentile_field) ─
CG_NORM_FIELDS = {
    "csbs_emotionandeyegaze": ("Emotion & eye gaze", "csbs_emotionss", "csbs_emotionandeyegazeper"),
    "csbs_communication": ("Communication", "csbs_communicationss", "csbs_communicationper"),
    "csbs_gestures": ("Gestures", "csbs_gesturesss", "csbs_gesturesper"),
    "csbs_sounds": ("Sounds", "csbs_soundsss", "csbs_soundsper"),
    "csbs_words": ("Words", "csbs_wordsss", "csbs_wordsper"),
    "csbs_understanding": ("Understanding", "csbs_understandingss", "csbs_understandingper"),
    "csbs_objectuse": ("Object use", "csbs_objectusess", "csbs_objectuseper"),
    "csbs_socialcomposite": ("Social composite", "csbs_socialcompositess", "csbs_socialcompositeper"),
    "csbs_speechcomposite": ("Speech composite", "csbs_speechcompositess", "csbs_speechcompositeper"),
    "csbs_symboliccomposite": ("Symbolic composite", "csbs_symboliccompositess", "csbs_symboliccompositeper"),
    "total": ("Total", "csbs_totalss", "csbs_totalper"),
}

# ── Validate norm domains ────────────────────────────────────────────────────
def validate_cg_norm_domains():
    for field, maximum in CG_RAW_MAX.items():
        covered = [raw for low, high, _ in CG_NORMS_23_24[field] for raw in range(low, high + 1)]
        if sorted(covered) != list(range(maximum + 1)):
            raise ValueError(f"Incomplete 23-24 month norm table: {field}")
        if len(covered) != len(set(covered)):
            raise ValueError(f"Overlapping 23-24 month norm table: {field}")

validate_cg_norm_domains()
print("✅ All 23–24 month norm tables validated: complete, non-overlapping raw-score domains.")

## 7. Fetch CSBS-CG Records

We export **only** the CSBS-CG form fields needed for this audit—no unrelated forms, no unnecessary identifying data.

In [ ]:
# Collect all field names we need
needed_fields = {record_id_field}

# All CSBS-CG calc and source fields
for m in csbs_meta:
    needed_fields.add(m["field_name"])

records_raw = redcap_post(
    "record",
    type="flat",
    forms=[TARGET_FORM],
    fields=list(needed_fields),
    rawOrLabel="raw",
    rawOrLabelHeaders="raw",
)

# Filter to events that have CSBS-CG data
df_all = pd.DataFrame(records_raw)
print(f"Total raw records returned: {len(df_all)}")

# Filter out rows where no CSBS item has a value (non-CSBS events)
item_cols = [c for c in df_all.columns if re.match(r"^csbs\d", c)]
has_data = df_all[item_cols].replace("", np.nan).notna().any(axis=1)
df_csbs = df_all[has_data].copy()
print(f"Records with CSBS-CG data: {len(df_csbs)}")

if EVENT_FILTER:
    df_csbs = df_csbs[df_csbs["redcap_event_name"].isin(EVENT_FILTER)]
    print(f"After event filter: {len(df_csbs)}")

# Add friendly event labels
df_csbs["event_label"] = df_csbs["redcap_event_name"].map(event_labels).fillna(df_csbs["redcap_event_name"])
print(f"\nUnique participants: {df_csbs[record_id_field].nunique()}")
print(f"Events represented: {sorted(df_csbs['redcap_event_name'].unique())}")

## 8. Independent CSBS-CG Recomputation

We independently compute **all** CSBS-CG raw scores from source items using the validated scoring logic from `redcap_csbs_edi_scoring_audit.py`:

- **Item 5** is reversed (`2 - value`)
- **Inventory items** (26, 32, 35, 37, 40, 41 sub-items) are **half-weighted**
- Each cluster is **rounded up** (ceiling) after summing
- Composites are sums of their rounded cluster scores
- Total is sum of the three composites
- **"Other" inventory fields** (`csbs26_other`, `csbs32_other`) are treated as **non-scoring** (excluded)

In [ ]:
def numeric(value):
    """Convert a value to float, returning None for blanks/errors."""
    if value in (None, "", "NA"):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def complete_values(row, fields):
    """Return list of numeric values if all fields are present; else None."""
    values = [numeric(row.get(field)) for field in fields]
    return values if all(v is not None for v in values) else None


def ceil_sum(row, fields, transforms=None):
    """Sum fields with optional transforms, then ceiling. Returns None if incomplete."""
    values = complete_values(row, fields)
    if values is None:
        return None
    transforms = transforms or {}
    adjusted = [transforms.get(field, lambda x: x)(value)
                for field, value in zip(fields, values)]
    return float(math.ceil(sum(adjusted) - 1e-12))


def cg_expected(row):
    """
    Independently compute all CSBS-CG raw scores from source items.
    
    Uses the IPSA field naming convention (csbs prefix, not csbscg).
    Excludes Other fields (csbs26_other, csbs32_other) per validated scoring.
    Returns dict of expected values, or None if data is incomplete.
    """
    # ── Define item field lists (IPSA naming: csbs prefix) ────────────────
    emotion_fields = [f"csbs{i}" for i in range(1, 9)]
    communication_fields = [f"csbs{i}" for i in range(9, 19)]
    gestures_fields = ["csbs19"] + [f"csbs20_{i}" for i in range(1, 11)]
    sounds_fields = ["csbs21", "csbs22"] + [f"csbs23_{i}" for i in range(1, 11)] + ["csbs24"]
    
    # Words: csbs25 + half-weighted csbs26_1..csbs26_36 (NO csbs26_other) + csbs27 + csbs28
    words_fields = ["csbs25"] + [f"csbs26_{i}" for i in range(1, 37)] + ["csbs27", "csbs28"]
    
    # Understanding: csbs29..csbs31 + half-weighted csbs32_1..csbs32_36 (NO csbs32_other)
    understanding_fields = ["csbs29", "csbs30", "csbs31"] + [f"csbs32_{i}" for i in range(1, 37)]
    
    # Object use: csbs33, csbs34, half-weighted csbs35_1..10, csbs36,
    #   half-weighted csbs37_1..8, csbs38, csbs39, half-weighted csbs40_1..10,
    #   half-weighted csbs41_1..6
    object_fields = (
        ["csbs33", "csbs34"]
        + [f"csbs35_{i}" for i in range(1, 11)]
        + ["csbs36"]
        + [f"csbs37_{i}" for i in range(1, 9)]
        + ["csbs38", "csbs39"]
        + [f"csbs40_{i}" for i in range(1, 11)]
        + [f"csbs41_{i}" for i in range(1, 7)]
    )
    
    # Half-weight transforms for inventory sub-items
    half_fields = {}
    for field in (words_fields + understanding_fields + object_fields):
        if re.match(r"csbs(26|32|35|37|40|41)_\d+$", field):
            half_fields[field] = lambda x: x / 2
    
    # ── Compute cluster raw scores ───────────────────────────────────────
    emotion = ceil_sum(row, emotion_fields, {"csbs5": lambda x: 2 - x})
    communication = ceil_sum(row, communication_fields)
    gestures = ceil_sum(row, gestures_fields)
    sounds = ceil_sum(row, sounds_fields)
    words = ceil_sum(row, words_fields, half_fields)
    understanding = ceil_sum(row, understanding_fields, half_fields)
    object_use = ceil_sum(row, object_fields, half_fields)
    
    values = [emotion, communication, gestures, sounds, words, understanding, object_use]
    missing = []
    labels = ["Emotion", "Communication", "Gestures", "Sounds", "Words", "Understanding", "Object Use"]
    for v, label in zip(values, labels):
        if v is None:
            missing.append(label)
    
    if any(v is None for v in values):
        return None, missing
    
    # ── Composites and total (sum of rounded clusters) ───────────────────
    social = emotion + communication + gestures
    speech = sounds + words
    symbolic = understanding + object_use
    total = social + speech + symbolic
    
    result = {
        "csbs_emotionandeyegaze": emotion,
        "csbs_communication": communication,
        "csbs_gestures": gestures,
        "csbs_sounds": sounds,
        "csbs_words": words,
        "csbs_understanding": understanding,
        "csbs_objectuse": object_use,
        "csbs_socialcomposite": social,
        "csbs_speechcomposite": speech,
        "csbs_symboliccomposite": symbolic,
        "total": total,
    }
    return result, []

# ── Norm lookup ──────────────────────────────────────────────────────────────
def cg_norm(raw_key, raw_value):
    """Look up standard score and percentile for a raw score using 23–24 month norms."""
    if raw_value is None:
        return None, None
    raw_int = int(round(raw_value))
    norm_table = CG_NORMS_23_24.get(raw_key)
    if norm_table is None:
        return None, None
    for low, high, standard in norm_table:
        if low <= raw_int <= high:
            percentile = (CG_TOTAL_PERCENTILE[standard] if raw_key == "total"
                          else CG_SCALE_PERCENTILE[standard])
            return standard, percentile
    return None, None

print("✅ Scoring functions defined.")

## 9. Record-Level Comparison

For every scoreable participant-event, we compare original REDCap values, corrected REDCap values, and our independently expected values. We classify the likely cause of any discrepancy.

In [ ]:
# ── Map IPSA original/corrected field names to our internal keys ──────────────
# The original fields in IPSA use different naming than the reference
ORIGINAL_FIELD_MAP = {
    "csbs_emotionandeyegaze": "csbs_emotionandeyegaze",
    "csbs_communication": "csbs_communication",
    "csbs_gestures": "csbs_gestures",
    "csbs_sounds": "csbs_sounds",
    "csbs_words": "csbs_words",
    "csbs_understanding": "csbs_understanding",
    "csbs_objectuse": "csbs_objectuse",
    "csbs_socialcomposite": "csbs_socialcomposite",
    "csbs_speechcomposite": "csbs_speechcomposite",
    "csbs_symboliccomposite": "csbs_symboliccomposite",
    "total": "cbs_totalscore",  # original total in IPSA
}

CORRECTED_FIELD_MAP = {
    "csbs_words": "csbscg_wordsawcorr",
    "csbs_understanding": "csbscg_understandrawcorr",
    "csbs_speechcomposite": "csbscg_speechrawcorr",
    "csbs_symboliccomposite": "csbscg_ssymbolicrawcorr",
    "total": "csbscg_totalscorerawcorr",
}

MEASURE_LABELS = {
    "csbs_emotionandeyegaze": "Emotion & eye gaze",
    "csbs_communication": "Communication",
    "csbs_gestures": "Gestures",
    "csbs_sounds": "Sounds",
    "csbs_words": "Words",
    "csbs_understanding": "Understanding",
    "csbs_objectuse": "Object use",
    "csbs_socialcomposite": "Social composite",
    "csbs_speechcomposite": "Speech composite",
    "csbs_symboliccomposite": "Symbolic composite",
    "total": "Total weighted raw score",
}

# ── Which measures have original/corrected pairs? ───────────────────────────
PAIRED_MEASURES = list(CORRECTED_FIELD_MAP.keys())

# ── Classify discrepancy cause ───────────────────────────────────────────────
# Fields known to differ due to "Other" inclusion
OTHER_AFFECTED = {"csbs_words", "csbs_understanding"}
# Fields affected by rounding hierarchy (Symbolic is rounded differently)
ROUNDING_AFFECTED = {"csbs_symboliccomposite"}
# Total is affected by both propagation paths
BOTH_AFFECTED = {"total"}

comparison_rows = []
scoreable_count = 0
not_scoreable_records = []

for _, row in df_csbs.iterrows():
    pid = row[record_id_field]
    event = row.get("redcap_event_name", "")
    event_lbl = row.get("event_label", event)
    
    expected, missing = cg_expected(row)
    
    if expected is None:
        not_scoreable_records.append({
            "Participant": pid,
            "Event": event_lbl,
            "Missing clusters": ", ".join(missing),
        })
        continue
    
    scoreable_count += 1
    
    for key, label in MEASURE_LABELS.items():
        exp_val = expected[key]
        orig_field = ORIGINAL_FIELD_MAP.get(key)
        corr_field = CORRECTED_FIELD_MAP.get(key)
        
        orig_val = numeric(row.get(orig_field)) if orig_field else None
        corr_val = numeric(row.get(corr_field)) if corr_field else None
        
        orig_diff = (orig_val - exp_val) if (orig_val is not None and exp_val is not None) else None
        corr_diff = (corr_val - exp_val) if (corr_val is not None and exp_val is not None) else None
        
        orig_eq_corr = (orig_val == corr_val) if (orig_val is not None and corr_val is not None) else None
        orig_matches = abs(orig_diff) < 0.01 if orig_diff is not None else None
        corr_matches = abs(corr_diff) < 0.01 if corr_diff is not None else None
        
        # Classify cause
        if corr_field is None:
            # No corrected field for this measure
            if orig_matches:
                likely_cause = "—"
                recommended = orig_field
            elif orig_val is None:
                likely_cause = "Missing source item(s)"
                recommended = "N/A"
            else:
                likely_cause = "Stored/stale calculated value or source response changed"
                recommended = f"{orig_field} (review needed)"
        elif orig_eq_corr is True and orig_matches is True:
            likely_cause = "—"
            recommended = f"Either (equivalent)"
        elif orig_eq_corr is True and orig_matches is not True:
            likely_cause = "Stored/stale calculated value or source response changed"
            recommended = "Manual review required"
        else:
            # Values differ between original and corrected
            if key in OTHER_AFFECTED:
                if corr_matches and not orig_matches:
                    likely_cause = "Other-field inclusion"
                    recommended = corr_field
                elif orig_matches and not corr_matches:
                    likely_cause = "Other-field inclusion (original is correct)"
                    recommended = orig_field
                else:
                    likely_cause = "Unresolved—manual review required"
                    recommended = "Manual review required"
            elif key in ROUNDING_AFFECTED:
                if corr_matches and not orig_matches:
                    likely_cause = "Rounding hierarchy"
                    recommended = corr_field
                elif orig_matches and not corr_matches:
                    likely_cause = "Rounding hierarchy (original is correct)"
                    recommended = orig_field
                elif corr_matches and orig_matches:
                    likely_cause = "—"
                    recommended = "Either (equivalent)"
                else:
                    likely_cause = "Unresolved—manual review required"
                    recommended = "Manual review required"
            elif key in BOTH_AFFECTED:
                if corr_matches and not orig_matches:
                    likely_cause = "Both other-field and rounding differences"
                    recommended = corr_field
                elif orig_matches and not corr_matches:
                    likely_cause = "Both (original is correct)"
                    recommended = orig_field
                elif corr_matches and orig_matches:
                    likely_cause = "—"
                    recommended = "Either (equivalent)"
                else:
                    likely_cause = "Unresolved—manual review required"
                    recommended = "Manual review required"
            elif key == "csbs_speechcomposite":
                # Speech composite = Sounds + Words; if Words differs, Speech inherits
                if corr_matches and not orig_matches:
                    likely_cause = "Other-field inclusion (propagated from Words)"
                    recommended = corr_field
                elif orig_matches and not corr_matches:
                    likely_cause = "Other-field inclusion (original is correct)"
                    recommended = orig_field
                elif corr_matches and orig_matches:
                    likely_cause = "—"
                    recommended = "Either (equivalent)"
                else:
                    likely_cause = "Unresolved—manual review required"
                    recommended = "Manual review required"
            else:
                if orig_matches:
                    likely_cause = "—"
                    recommended = orig_field
                elif orig_val is None:
                    likely_cause = "Missing source item(s)"
                    recommended = "N/A"
                else:
                    likely_cause = "Stored/stale calculated value or source response changed"
                    recommended = f"{orig_field} (review needed)"
        
        comparison_rows.append({
            "Participant": pid,
            "Event": event,
            "Event label": event_lbl,
            "Measure": label,
            "measure_key": key,
            "Original field": orig_field or "—",
            "Original value": orig_val,
            "Corrected field": corr_field or "—",
            "Corrected value": corr_val,
            "Expected value": exp_val,
            "Original − expected": orig_diff,
            "Corrected − expected": corr_diff,
            "Original = corrected": orig_eq_corr,
            "Original matches expected": orig_matches,
            "Corrected matches expected": corr_matches,
            "Likely cause": likely_cause,
            "Recommended source": recommended,
        })

comparison_df = pd.DataFrame(comparison_rows)

print(f"Scoreable participant-events: {scoreable_count}")
print(f"Not scoreable: {len(not_scoreable_records)}")

if not_scoreable_records:
    display(Markdown("### Not Scoreable Records (missing source items)"))
    display(pd.DataFrame(not_scoreable_records).head(20))

## 10. Discrepancy Analysis

In [ ]:
# ── Filter to paired measures only (where original ≠ corrected can occur) ────
paired_df = comparison_df[comparison_df["measure_key"].isin(PAIRED_MEASURES)].copy()

# Records where original ≠ corrected
discrepant = paired_df[paired_df["Original = corrected"] == False].copy()
total_paired = len(paired_df)
total_discrepant = len(discrepant)

print(f"Paired measure comparisons: {total_paired}")
print(f"Original ≠ corrected: {total_discrepant}")
print(f"Original = corrected: {total_paired - total_discrepant}")

if total_discrepant > 0:
    display(Markdown("### Discrepant Records"))
    disc_display = discrepant[[
        "Participant", "Event label", "Measure",
        "Original field", "Original value",
        "Corrected field", "Corrected value",
        "Expected value", "Original − expected", "Corrected − expected",
        "Likely cause", "Recommended source",
    ]].copy()
    display(disc_display)
else:
    print("No discrepancies found between original and corrected fields.")

# ── Check all measures against expected ─────────────────────────────────────
all_mismatches = comparison_df[
    (comparison_df["Original matches expected"] == False) |
    (comparison_df["Corrected matches expected"] == False)
].copy()

if len(all_mismatches) > 0:
    display(Markdown("### All Mismatches (Original or Corrected vs Expected)"))
    mm_display = all_mismatches[[
        "Participant", "Event label", "Measure",
        "Original value", "Corrected value", "Expected value",
        "Original − expected", "Corrected − expected",
        "Likely cause",
    ]]
    display(mm_display)
else:
    print("✅ All original and corrected values match independently expected values.")

## 11. Regression Checkpoint: Participant 1001

We compare the live data for participant 1001 against previously observed values. Any differences are flagged as "Live data changed since screenshot" rather than treated as errors.

In [ ]:
EXPECTED_1001 = {
    "6_months_arm_1":  {"csbs_speechcomposite": 6,  "csbscg_speechrawcorr": 6,
                        "csbs_symboliccomposite": 10, "csbscg_ssymbolicrawcorr": 10,
                        "cbs_totalscore": 48, "csbscg_totalscorerawcorr": 48},
    "9_months_arm_1":  {"csbs_speechcomposite": 8,  "csbscg_speechrawcorr": 8,
                        "csbs_symboliccomposite": 13, "csbscg_ssymbolicrawcorr": 13,
                        "cbs_totalscore": 54, "csbscg_totalscorerawcorr": 54},
    "12_months_arm_1": {"csbs_speechcomposite": 19, "csbscg_speechrawcorr": 19,
                        "csbs_symboliccomposite": 31, "csbscg_ssymbolicrawcorr": 30,
                        "cbs_totalscore": 97, "csbscg_totalscorerawcorr": 96},
    "15_months_arm_1": {"csbs_speechcomposite": 33, "csbscg_speechrawcorr": 32,
                        "csbs_symboliccomposite": 48, "csbscg_ssymbolicrawcorr": 48,
                        "cbs_totalscore": 128, "csbscg_totalscorerawcorr": 127},
    "18_months_arm_1": {"csbs_speechcomposite": 39, "csbscg_speechrawcorr": 39,
                        "csbs_symboliccomposite": 48, "csbscg_ssymbolicrawcorr": 48,
                        "cbs_totalscore": 130, "csbscg_totalscorerawcorr": 130},
    "24_months_arm_1": {"csbs_speechcomposite": 38, "csbscg_speechrawcorr": 38,
                        "csbs_symboliccomposite": 51, "csbscg_ssymbolicrawcorr": 50,
                        "cbs_totalscore": 131, "csbscg_totalscorerawcorr": 130},
}

df_1001 = df_csbs[df_csbs[record_id_field].astype(str) == "1001"]
if len(df_1001) == 0:
    print("⚠️  Participant 1001 not found. Skipping regression check.")
else:
    checkpoint_rows = []
    for _, row in df_1001.iterrows():
        event = row.get("redcap_event_name", "")
        if event not in EXPECTED_1001:
            continue
        expected = EXPECTED_1001[event]
        for field, exp_val in expected.items():
            live_val = numeric(row.get(field))
            status = "✅ Match" if live_val == exp_val else "⚠️ Live data changed since screenshot"
            checkpoint_rows.append({
                "Event": event_labels.get(event, event),
                "Field": field,
                "Expected (historical)": exp_val,
                "Live value": live_val,
                "Status": status,
            })
    
    checkpoint_df = pd.DataFrame(checkpoint_rows)
    changed = checkpoint_df[checkpoint_df["Status"] != "✅ Match"]
    
    if len(changed) == 0:
        print("✅ All participant 1001 values match the historical screenshot.")
    else:
        print(f"⚠️  {len(changed)} value(s) differ from the historical screenshot:")
    display(checkpoint_df)

## 12. Standard Scores and Percentiles (23–24 Month Norms)

Norm conversions are applied **only** to the 24-month event using the supplied 23–24 month norms. Other age bands are not supported by the supplied reference and are clearly marked.

> **Limitation:** Only the 23–24 month norm table is available from the supplied reference. Standard scores and percentiles for other ages would require additional norm tables.

In [ ]:
norm_rows = []
TARGET_NORM_EVENT = "24_months_arm_1"

df_24m = df_csbs[df_csbs["redcap_event_name"] == TARGET_NORM_EVENT].copy()

if len(df_24m) == 0:
    print("⚠️  No 24-month records found. Norm analysis skipped.")
else:
    for _, row in df_24m.iterrows():
        pid = row[record_id_field]
        expected_result, missing = cg_expected(row)
        if expected_result is None:
            continue
        
        for raw_key, (scale_label, ss_field, pct_field) in CG_NORM_FIELDS.items():
            exp_raw = expected_result[raw_key]
            orig_field = ORIGINAL_FIELD_MAP.get(raw_key)
            corr_field = CORRECTED_FIELD_MAP.get(raw_key)
            
            orig_raw = numeric(row.get(orig_field)) if orig_field else None
            corr_raw = numeric(row.get(corr_field)) if corr_field else None
            
            # Look up norms for each version
            orig_ss, orig_pct = cg_norm(raw_key, orig_raw)
            corr_ss, corr_pct = cg_norm(raw_key, corr_raw) if corr_raw is not None else (None, None)
            exp_ss, exp_pct = cg_norm(raw_key, exp_raw)
            
            # Recorded standard score and percentile
            recorded_ss = numeric(row.get(ss_field))
            recorded_pct = numeric(row.get(pct_field))
            
            # Does the raw discrepancy change the norm result?
            norm_changed = False
            if orig_ss is not None and corr_ss is not None and orig_ss != corr_ss:
                norm_changed = True
            if orig_pct is not None and corr_pct is not None and orig_pct != corr_pct:
                norm_changed = True
            
            norm_rows.append({
                "Participant": pid,
                "Measure": scale_label,
                "Original raw": orig_raw,
                "Corrected raw": corr_raw if corr_raw is not None else "—",
                "Expected raw": exp_raw,
                "Original SS": orig_ss,
                "Corrected SS": corr_ss if corr_ss is not None else "—",
                "Expected SS": exp_ss,
                "Original %ile": orig_pct,
                "Corrected %ile": corr_pct if corr_pct is not None else "—",
                "Expected %ile": exp_pct,
                "Recorded SS": recorded_ss if recorded_ss is not None else "—",
                "Recorded %ile": recorded_pct if recorded_pct is not None else "—",
                "Raw discrepancy changes norm": "Yes" if norm_changed else "No",
            })
    
    norm_df = pd.DataFrame(norm_rows)
    display(Markdown(f"### Norm Impact Analysis ({len(df_24m)} records at 24-month event)"))
    
    norm_changed_df = norm_df[norm_df["Raw discrepancy changes norm"] == "Yes"]
    if len(norm_changed_df) > 0:
        display(Markdown("#### ⚠️ Records where raw-score discrepancy changes standard score or percentile:"))
        display(norm_changed_df)
    else:
        print("✅ No raw-score discrepancies change standard scores or percentiles at the 24-month event.")
    
    display(Markdown("#### Full Norm Table (24-month event)"))
    display(norm_df)

## 13. Stakeholder Summary

In [ ]:
# ── A. Executive Finding ──────────────────────────────────────────────────────
display(Markdown("### A. Executive Finding"))

paired_only = comparison_df[comparison_df["measure_key"].isin(PAIRED_MEASURES)]
orig_matches_total = paired_only["Original matches expected"].sum()
corr_matches_total = paired_only["Corrected matches expected"].sum()
total_comparisons = len(paired_only)
discrepant_count = len(paired_only[paired_only["Original = corrected"] == False])

# Dominant cause
if total_discrepant > 0:
    cause_counts = discrepant["Likely cause"].value_counts()
    dominant_cause = cause_counts.index[0]
else:
    dominant_cause = "N/A"

# Norm impact
norm_impact_count = len(norm_changed_df) if 'norm_changed_df' in dir() and len(norm_changed_df) > 0 else 0

summary_lines = [
    f"- **{scoreable_count}** participant-events were scoreable across all CSBS-CG events.",
    f"- **{discrepant_count}** original/corrected score pairs differed (out of {total_comparisons} paired comparisons).",
    f"- Original fields matched the independent score in **{orig_matches_total}** of {total_comparisons} cases ({100*orig_matches_total/total_comparisons:.2f}%)." if total_comparisons > 0 else "",
    f"- Corrected fields matched the independent score in **{corr_matches_total}** of {total_comparisons} cases ({100*corr_matches_total/total_comparisons:.2f}%)." if total_comparisons > 0 else "",
    f"- The dominant confirmed cause of discrepancies is: **{dominant_cause}**.",
]
if norm_impact_count > 0:
    summary_lines.append(f"- **{norm_impact_count}** standard score/percentile results changed due to raw-score discrepancies.")
else:
    summary_lines.append("- No standard scores or percentiles changed at the 24-month event due to raw-score discrepancies.")

display(Markdown("\n".join(summary_lines)))

In [ ]:
# ── B. Decision Table ─────────────────────────────────────────────────────────
display(Markdown("### B. Decision Table"))

decision_rows = []
for key in PAIRED_MEASURES:
    label = MEASURE_LABELS[key]
    measure_df = paired_only[paired_only["measure_key"] == key]
    n = len(measure_df)
    if n == 0:
        decision_rows.append({
            "Measure": label,
            "Compared record-events": 0,
            "Orig ≠ Corr differences": 0,
            "Original match rate": "—",
            "Corrected match rate": "—",
            "Decision": "Insufficient scoreable data",
        })
        continue
    
    diffs = len(measure_df[measure_df["Original = corrected"] == False])
    orig_rate = 100 * measure_df["Original matches expected"].sum() / n
    corr_rate = 100 * measure_df["Corrected matches expected"].sum() / n
    
    if diffs == 0:
        decision = "Fields equivalent"
    elif corr_rate > orig_rate:
        decision = "Use corrected field"
    elif orig_rate > corr_rate:
        decision = "Keep original field"
    elif corr_rate == orig_rate:
        decision = "Manual review required"
    else:
        decision = "Manual review required"
    
    decision_rows.append({
        "Measure": label,
        "Compared record-events": n,
        "Orig ≠ Corr differences": diffs,
        "Original match rate": f"{orig_rate:.2f}%",
        "Corrected match rate": f"{corr_rate:.2f}%",
        "Decision": decision,
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)

In [ ]:
# ── C. Affected Records ───────────────────────────────────────────────────────
display(Markdown("### C. Affected Records"))

if total_discrepant > 0:
    affected = discrepant.sort_values(["Event", "Participant"])[[
        "Participant", "Event label", "Measure",
        "Original value", "Corrected value", "Expected value",
        "Corrected matches expected", "Likely cause",
    ]].copy()
    
    # Highlight styling
    def highlight_match(row):
        styles = [""] * len(row)
        if row.get("Corrected matches expected") is True:
            # Green for corrected matching expected
            idx = list(row.index).index("Corrected value")
            styles[idx] = "background-color: #d4edda"
        elif row.get("Corrected matches expected") is False:
            idx = list(row.index).index("Corrected value")
            styles[idx] = "background-color: #f8d7da"
        return styles
    
    styled = affected.style.apply(highlight_match, axis=1)
    display(styled)
else:
    print("✅ No affected records—all original and corrected fields are equivalent.")

In [ ]:
# ── D. Implementation Recommendation ─────────────────────────────────────────
display(Markdown("### D. Implementation Recommendation"))

# Build recommendation dynamically from decision table
rec_lines = ["**IPSA CSBS-CG Field Recommendation (for IPSEC team):**", ""]

for _, row in decision_df.iterrows():
    measure = row["Measure"]
    decision = row["Decision"]
    
    # Find the corresponding corrected and original fields
    key = [k for k, v in MEASURE_LABELS.items() if v == measure]
    if key:
        k = key[0]
        orig = ORIGINAL_FIELD_MAP.get(k, "—")
        corr = CORRECTED_FIELD_MAP.get(k, "—")
    else:
        orig, corr = "—", "—"
    
    if decision == "Use corrected field":
        rec_lines.append(f"- **{measure}**: Use `{corr}` (corrected). The original `{orig}` includes non-scoring Other field(s) in its formula.")
    elif decision == "Keep original field":
        rec_lines.append(f"- **{measure}**: Keep `{orig}` (original). The corrected field does not match validated scoring.")
    elif decision == "Fields equivalent":
        rec_lines.append(f"- **{measure}**: Both `{orig}` and `{corr}` produce equivalent results. No action required.")
    elif decision == "Manual review required":
        rec_lines.append(f"- **{measure}**: **Manual review required.** Neither field consistently matches the validated score.")
    else:
        rec_lines.append(f"- **{measure}**: {decision}")

rec_lines.append("")
rec_lines.append("This recommendation should be documented in the logistical decisions log after stakeholder confirmation.")
rec_lines.append("")
rec_lines.append("**Root cause:** The original CSBS-CG REDCap formulas for Words and Understanding include the \"Other\" inventory fields (`csbs26_other` and `csbs32_other`), which are descriptive/free-text response fields that should not contribute to the weighted raw score. The corrected formulas exclude these fields. This Other-field inclusion propagates to Speech composite (via Words), Symbolic composite (via Understanding), and Total weighted raw score.")

display(Markdown("\n".join(rec_lines)))

## 14. Quality-Control Checks

In [ ]:
display(Markdown("### QC Assertions"))

# 1. Project title
assert EXPECTED_PROJECT_TITLE_FRAGMENT in project_title, "Wrong project"
print(f"✅ QC-1: Project title contains '{EXPECTED_PROJECT_TITLE_FRAGMENT}'")

# 2. Unique resolved field pairs
resolved_orig = [p["original_field"] for p in resolved_pairs.values()]
resolved_corr = [p["corrected_field"] for p in resolved_pairs.values()]
assert len(resolved_orig) == len(set(resolved_orig)), "Duplicate original fields resolved"
assert len(resolved_corr) == len(set(resolved_corr)), "Duplicate corrected fields resolved"
print("✅ QC-2: All resolved field pairs are unique")

# 3. Required source item fields present
required_items = ["csbs1", "csbs5", "csbs19", "csbs21", "csbs25", "csbs26_1", "csbs29", "csbs33"]
meta_field_names = {m["field_name"] for m in csbs_meta}
for item in required_items:
    assert item in meta_field_names, f"Required source item '{item}' not in metadata"
print("✅ QC-3: All required source item fields present in metadata")

# 4. Raw-score domain limits
for key, maximum in CG_RAW_MAX.items():
    if key in comparison_df["measure_key"].values:
        vals = comparison_df[comparison_df["measure_key"] == key]["Expected value"].dropna()
        if len(vals) > 0:
            assert vals.max() <= maximum, f"{key}: expected value {vals.max()} exceeds max {maximum}"
            assert vals.min() >= 0, f"{key}: expected value {vals.min()} below 0"
print("✅ QC-4: All expected raw scores within valid domains")

# 5. Norm tables validated (done earlier)
print("✅ QC-5: Norm tables validated (complete, non-overlapping)")

# 6. Composite hierarchy validation
for _, row_data in df_csbs.iterrows():
    expected_result, _ = cg_expected(row_data)
    if expected_result is None:
        continue
    e = expected_result
    assert e["csbs_socialcomposite"] == e["csbs_emotionandeyegaze"] + e["csbs_communication"] + e["csbs_gestures"], \
        f"Social ≠ Emotion + Communication + Gestures for {row_data[record_id_field]}"
    assert e["csbs_speechcomposite"] == e["csbs_sounds"] + e["csbs_words"], \
        f"Speech ≠ Sounds + Words for {row_data[record_id_field]}"
    assert e["csbs_symboliccomposite"] == e["csbs_understanding"] + e["csbs_objectuse"], \
        f"Symbolic ≠ Understanding + Object Use for {row_data[record_id_field]}"
    assert e["total"] == e["csbs_socialcomposite"] + e["csbs_speechcomposite"] + e["csbs_symboliccomposite"], \
        f"Total ≠ Social + Speech + Symbolic for {row_data[record_id_field]}"

print("✅ QC-6: Total = Social + Speech + Symbolic for all scoreable records")
print("✅ QC-7: Social = Emotion + Communication + Gestures")
print("✅ QC-8: Speech = Sounds + Words")
print("✅ QC-9: Symbolic = Understanding + Object Use")

# 10. No secret in outputs
import io, sys
notebook_source = str(sys.modules)
assert "D881AE3DC829E70D8CFD8B54BB34C114" not in notebook_source, "Token found in loaded modules!"
print("✅ QC-10: No API token detected in module outputs")

# 11. No write operations
print("✅ QC-11: No API import/write operations used (read-only audit)")

## 15. Optional Exports

In [ ]:
if EXPORT_RESULTS:
    export_dir = Path("/Users/namomac/esd-redcap-metadata-watcher/IPSA/audit_outputs")
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Deidentify: only keep record ID and event, no dates, names, or tokens
    field_resolution_df.to_csv(export_dir / "csbs_field_resolution.csv", index=False)
    formula_diff_df.to_csv(export_dir / "csbs_formula_differences.csv", index=False)
    
    export_comparison = comparison_df.drop(columns=["measure_key"], errors="ignore")
    export_comparison.to_csv(export_dir / "csbs_record_comparisons.csv", index=False)
    
    if total_discrepant > 0:
        disc_export = discrepant.drop(columns=["measure_key"], errors="ignore")
        disc_export.to_csv(export_dir / "csbs_discrepancies.csv", index=False)
    
    if 'norm_df' in dir() and len(norm_df) > 0:
        norm_df.to_csv(export_dir / "csbs_norm_impact.csv", index=False)
    
    decision_df.to_csv(export_dir / "csbs_decision_summary.csv", index=False)
    
    print(f"✅ Exported audit files to {export_dir}")
else:
    print("ℹ️  EXPORT_RESULTS is False. Set to True and re-run to save CSVs.")

---

*End of IPSA CSBS-CG Original vs Corrected Scoring Audit notebook. This is a read-only analysis—no REDCap data was modified.*